In [5]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('/Users/sricharanreddykonyala/Documents/Projects/SQLGuard/data/raw/SQLiV3.csv')

In [6]:
df_sqli = pd.read_csv(DATA_PATH)

print("Dataset shape:", df_sqli.shape)
print("\nColumns:")
print(df_sqli.columns.tolist())

display(df_sqli.head())

Dataset shape: (30919, 2)

Columns:
['Query', 'Label']


,Query,Label
0,""" or pg_sleep ( __TIME__ ) --",1
1,create user name identified by pass123 tempora...,1
2,AND 1 = utl_inaddr.get_host_address ( ...,1
3,select * from users where id = '1' or @ @1 ...,1
4,"select * from users where id = 1 or 1#"" ( ...",1


In [ ]:
print("Dataset shape:", df_sqli.shape)

print("\nData types:")
print(df_sqli.dtypes)

print("\nLabel distribution:")
display(df_sqli["Label"].value_counts(dropna=False))

print("\nLabel proportions:")
display(
    df_sqli["Label"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("\nMissing values:")
display(df_sqli.isna().sum())

print("\nExact duplicate rows:", df_sqli.duplicated().sum())

print("Duplicate queries:", df_sqli["Query"].duplicated().sum())

print("Queries with conflicting labels:",
    (
        df_sqli.groupby("Query", dropna=False)["Label"]
        .nunique()
        .gt(1)
    ).sum()
)

Dataset shape: (30919, 2)

Data types:
Query      str
Label    int64
dtype: object

Label distribution:


Label
0    19537
1    11382
Name: count, dtype: int64


Label proportions:


Label
0    63.19
1    36.81
Name: proportion, dtype: float64


Missing values:


Query    0
Label    0
dtype: int64


Exact duplicate rows: 12
Duplicate queries: 14
Queries with conflicting labels: 2


In [ ]:
label_counts = (df_sqli.groupby("Query", dropna=False)["Label"].nunique()
)

conflicting_queries = label_counts[label_counts > 1].index

print("Conflicting queries:", len(conflicting_queries))

display(
    df_sqli[df_sqli["Query"].isin(conflicting_queries)]
    .sort_values("Query")
)

Conflicting queries: 2


,Query,Label
656,#NAME?,1
866,#NAME?,1
13318,#NAME?,0
13532,#NAME?,0
15635,#NAME?,0
17913,#NAME?,0
18233,#NAME?,0
18875,#NAME?,0
4244,1,1
17499,1,0


In [9]:
# Preserve the original dataframe
df_clean = df_sqli.copy()

# Exclude query strings with conflicting labels
df_clean = df_clean[
    ~df_clean["Query"].isin(conflicting_queries)
].copy()

print("Original rows:", len(df_sqli))
print("Cleaned rows:", len(df_clean))
print("Rows excluded:", len(df_sqli) - len(df_clean))

print("\nRemaining conflicting labels:")
print(
    df_clean.groupby("Query")["Label"]
    .nunique()
    .gt(1)
    .sum()
)

Original rows: 30919
Cleaned rows: 30909
Rows excluded: 10

Remaining conflicting labels:
0


In [13]:
print("--- Label 0 Examples ---")
for i, q in enumerate(
    df_clean[df_clean["Label"] == 0].sample(n=5, random_state=42)["Query"],
    1,
):
  print(f"{i}. {q}")

print("\n--- Label 1 Examples ---")
for i, q in enumerate(
    df_clean[df_clean["Label"] == 1].sample(n=5, random_state=42)["Query"],
    1,
):
  print(f"{i}. {q}")

--- Label 0 Examples ---
1. SELECT * FROM offer FETCH FIRST 50 PERCENT ROWS ONLYSELECT TOP 3 * FROM dawn
2. SELECT * FROM watch WHERE out BETWEEN '1996-07-01' AND '1996-07-31'
3. 25585710g
4. 1pa5364e3a
5. SELECT * FROM slave

--- Label 1 Examples ---
1. 1"   )    )    as eqeb where 6784  =  6784 and updatexml  (  3393,concat  (  0x2e,0x7171706a71,  (  select   (  elt  (  3393  =  3393,1   )    )     )  ,0x717a767a71  )  ,1161  )  --
2. 1' and   (  3020  =  3020  )  *6703
3. 1"  )   where 1807  =  1807
4. 1'  )   as ccee where 8880  =  8880 and 8514  =    (  select count  (  *  )   from domain.domains as t1,domain.columns as t2,domain.tables as t3  )  --
5. 1"   )    )     )   or 5286  =    (  select count  (  *  )   from all_users t1,all_users t2,all_users t3,all_users t4,all_users t5  )   and    (    (     (  "ddun"  =  "ddun


In [14]:
print("Total rows:", len(df_clean))
print("Unique query strings:", df_clean["Query"].nunique())
print("Repeated query rows:", df_clean["Query"].duplicated().sum())

print("\nRows per label:")
display(df_clean["Label"].value_counts())

print("\nQuery lengths by label:")
display(
    df_clean.assign(query_length=df_clean["Query"].str.len())
    .groupby("Label")["query_length"]
    .describe()
    .round(2)
)

Total rows: 30909
Unique query strings: 30903
Repeated query rows: 6

Rows per label:


Label
0    19530
1    11379
Name: count, dtype: int64


Query lengths by label:


,count,mean,std,min,25%,50%,75%,max
Label,,,,,,,,
0,19530.0,39.92,51.75,1.0,11.0,35.0,56.0,5370.0
1,11379.0,118.64,96.83,1.0,51.0,85.0,150.0,456.0


In [15]:
from sklearn.model_selection import train_test_split

# One row per unique query
unique_queries = df_clean.drop_duplicates(subset="Query").copy()

# Split unique queries into training and testing
train_df, test_df = train_test_split(
    unique_queries,
    test_size=0.20,
    random_state=42,
    stratify=unique_queries["Label"]
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining label distribution:")
display(train_df["Label"].value_counts(normalize=True).round(3))

print("\nTesting label distribution:")
display(test_df["Label"].value_counts(normalize=True).round(3))

# Verify no exact query overlap
overlap = set(train_df["Query"]) & set(test_df["Query"])
print("\nShared queries between splits:", len(overlap))

Training rows: 24722
Testing rows: 6181

Training label distribution:


Label
0    0.632
1    0.368
Name: proportion, dtype: float64


Testing label distribution:


Label
0    0.632
1    0.368
Name: proportion, dtype: float64


Shared queries between splits: 0


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=50_000,
    dtype=np.float32,
    sublinear_tf=True
)

X_train = tfidf.fit_transform(train_df["Query"])
X_test = tfidf.transform(test_df["Query"])

y_train = train_df["Label"].to_numpy()
y_test = test_df["Label"].to_numpy()

print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)
print("Data type:", X_train.dtype)
print("Training matrix density:",
      round(X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100, 4),
      "%")

Training feature matrix: (24722, 50000)
Testing feature matrix: (6181, 50000)
Data type: float32
Training matrix density: 0.2542 %


In [17]:
from sklearn.linear_model import LogisticRegression
import time

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

start_time = time.time()

model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")
print("Model trained successfully!")

Training time: 0.30 seconds
Model trained successfully!


In [18]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Predict labels for unseen test queries
y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Benign (0)", "Attack (1)"],
        zero_division=0
    )
)

print("\nSummary:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")

Confusion Matrix:
[[3904    2]
 [  41 2234]]

Classification Report:
              precision    recall  f1-score   support

  Benign (0)       0.99      1.00      0.99      3906
  Attack (1)       1.00      0.98      0.99      2275

    accuracy                           0.99      6181
   macro avg       0.99      0.99      0.99      6181
weighted avg       0.99      0.99      0.99      6181


Summary:
Accuracy:  0.9930
Precision: 0.9991
Recall:    0.9820
F1-score:  0.9905


In [19]:
# Combine test queries, actual labels, and predictions
results = test_df[["Query", "Label"]].copy()
results["Predicted"] = y_pred

# False negatives: attacks predicted as benign
false_negatives = results[
    (results["Label"] == 1) &
    (results["Predicted"] == 0)
]

# False positives: benign queries predicted as attacks
false_positives = results[
    (results["Label"] == 0) &
    (results["Predicted"] == 1)
]

print("False negatives:", len(false_negatives))
print("\nMissed attack examples:")
print(false_negatives[["Query", "Label", "Predicted"]].head(10).to_string(index=False))

print("\nFalse positives:", len(false_positives))
print("\nBenign queries incorrectly flagged:")
print(false_positives[["Query", "Label", "Predicted"]].to_string(index=False))

False negatives: 41

Missed attack examples:
                                                                                     Query  Label  Predicted
                                                                                       ||6      1          0
                                                                                       1%'      1          0
                                                              or 'unusual'   =   'unusual'      1          0
create user name identified by pass123 temporary tablespace temp default tablespace users;      1          0
                                                                      or 2 between 1 and 3      1          0
                                                                                       asc      1          0
                                                                                       &1c      1          0
                                                                                   

In [20]:
# Probability that each test query belongs to class 1 (attack)
attack_probs = model.predict_proba(X_test)[:, 1]

results = test_df[["Query", "Label"]].copy()
results["Predicted"] = y_pred
results["Attack_Probability"] = attack_probs

false_negatives = results[
    (results["Label"] == 1) &
    (results["Predicted"] == 0)
].sort_values("Attack_Probability")

print("Most confidently missed attacks:")
print(
    false_negatives[
        ["Query", "Attack_Probability"]
    ].head(10).to_string(index=False)
)

print("\nLeast confidently missed attacks:")
print(
    false_negatives[
        ["Query", "Attack_Probability"]
    ].tail(10).sort_values(
        "Attack_Probability", ascending=False
    ).to_string(index=False)
)

Most confidently missed attacks:
                                         Query  Attack_Probability
                                    1;SELECT *            0.034408
grant connect to name; grant resource to name;            0.035684
                                          desc            0.053190
                                           asc            0.066346
                                        insert            0.069995
                                           UEs            0.085204
                                     5739-5738            0.096539
                                             |            0.097196
                                            //            0.097196
                                           ||6            0.097196

Least confidently missed attacks:
                                                 Query  Attack_Probability
                                  or 2 between 1 and 3            0.443472
                                       (  sel

In [21]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    y_threshold = (attack_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test, y_threshold, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_threshold, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, y_threshold, zero_division=0
        ),
        "False Positives": int(
            ((y_test == 0) & (y_threshold == 1)).sum()
        ),
        "False Negatives": int(
            ((y_test == 1) & (y_threshold == 0)).sum()
        )
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


,Threshold,Precision,Recall,F1-score,False Positives,False Negatives
0,0.10,0.8813,0.9890,0.9321,303,25
1,0.15,0.9770,0.9886,0.9827,53,26
2,0.20,0.9886,0.9873,0.9879,26,29
3,0.25,0.9938,0.9864,0.9901,14,31
4,0.30,0.9956,0.9846,0.9901,10,35
5,0.40,0.9987,0.9837,0.9911,3,37
6,0.50,0.9991,0.9820,0.9905,2,41


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Split the existing training data into sub-training and validation sets
subtrain_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    stratify=train_df["Label"]
)

# Learn text features using sub-training data only
tfidf_val = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=50_000,
    dtype=np.float32,
    sublinear_tf=True
)

X_subtrain = tfidf_val.fit_transform(subtrain_df["Query"])
X_val = tfidf_val.transform(val_df["Query"])

y_subtrain = subtrain_df["Label"].to_numpy()
y_val = val_df["Label"].to_numpy()

# Train a fresh model
model_val = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_val.fit(X_subtrain, y_subtrain)

# Compare training and validation performance
pred_train = model_val.predict(X_subtrain)
pred_val = model_val.predict(X_val)

print("SUB-TRAIN PERFORMANCE")
print(classification_report(y_subtrain, pred_train, zero_division=0))

print("VALIDATION PERFORMANCE")
print(classification_report(y_val, pred_val, zero_division=0))

print("Validation confusion matrix:")
print(confusion_matrix(y_val, pred_val))

SUB-TRAIN PERFORMANCE
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     12496
           1       1.00      0.99      0.99      7281

    accuracy                           1.00     19777
   macro avg       1.00      0.99      1.00     19777
weighted avg       1.00      1.00      1.00     19777

VALIDATION PERFORMANCE
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3125
           1       1.00      0.99      0.99      1820

    accuracy                           1.00      4945
   macro avg       1.00      0.99      1.00      4945
weighted avg       1.00      1.00      1.00      4945

Validation confusion matrix:
[[3125    0]
 [  20 1800]]


In [23]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

val_attack_probs = model_val.predict_proba(X_val)[:, 1]

threshold_results_val = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (val_attack_probs >= threshold).astype(int)

    threshold_results_val.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, pred, zero_division=0
        ),
        "False Positives": int(
            ((y_val == 0) & (pred == 1)).sum()
        ),
        "False Negatives": int(
            ((y_val == 1) & (pred == 0)).sum()
        )
    })

display(pd.DataFrame(threshold_results_val).round(4))

,Threshold,Precision,Recall,F1-score,False Positives,False Negatives
0,0.1,0.8278,0.9984,0.9051,378,3
1,0.2,0.9869,0.9951,0.9910,24,9
2,0.3,0.9978,0.9929,0.9953,4,13
3,0.4,1.0000,0.9912,0.9956,0,16
4,0.5,1.0000,0.9890,0.9945,0,20
5,0.6,1.0000,0.9857,0.9928,0,26
6,0.7,1.0000,0.9835,0.9917,0,30


In [25]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Transform test queries using the validation model's TF-IDF vectorizer
X_test_val = tfidf_val.transform(test_df["Query"])

# Predict attack probabilities
test_attack_probs = model_val.predict_proba(X_test_val)[:, 1]

# Apply the threshold selected using validation data
test_pred = (test_attack_probs >= 0.30).astype(int)

print("Test confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nTest classification report:")
print(classification_report(
    y_test,
    test_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

print(f"Test accuracy: {accuracy_score(y_test, test_pred):.4f}")

Test confusion matrix:
[[3896   10]
 [  35 2240]]

Test classification report:
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99      3906
      Attack       1.00      0.98      0.99      2275

    accuracy                           0.99      6181
   macro avg       0.99      0.99      0.99      6181
weighted avg       0.99      0.99      0.99      6181

Test accuracy: 0.9927


# SQLiV3 Dataset Exploration and Baseline Model

## Objective
Explore SQLiV3 and develop a supervised machine learning baseline for classifying SQL-related inputs as benign or attack-related.

## Dataset Overview
- **Original rows:** 30,919
- **Columns:** `Query`, `Label`
- **Labels:** `0` — benign, `1` — attack-related
- **Class distribution:** 63.19% benign and 36.81% attack-related.

## Data Cleaning and Preparation
- Removed exact duplicate rows.
- Excluded 10 rows associated with conflicting query labels.
- Created a cleaned dataset containing 30,909 rows.
- Removed repeated query strings before splitting to prevent exact query overlap.
- Created stratified training and test sets: 24,722 and 6,181 unique queries, respectively.

## Feature Engineering and Model
Character-level TF-IDF was used to represent query text, with 3–5 character n-grams and a maximum of 50,000 features.

A Logistic Regression classifier with balanced class weights was trained as the initial supervised baseline.

## Evaluation
A validation split was used to compare classification thresholds. A threshold of `0.30` was provisionally selected to balance missed attacks and false positives.

Preliminary held-out test results at this threshold:

| Metric | Result |
|---|---:|
| Accuracy | 99.27% |
| Attack precision | 99.56% |
| Attack recall | 98.46% |
| Attack F1-score | 99.01% |
| False positives | 10 |
| False negatives | 35 |

## Conclusion
The supervised baseline showed promising performance on SQLiV3 and is suitable for initial integration into SQLGuard's FastAPI backend.

## Limitations
- Dataset labels may contain errors.
- Similar payload patterns may occur across splits.
- Independent evaluation is required before making real-world security claims.